# Gerchberg-Saxton (GS) Reconstruction Script

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class AngularSpectrumPropagator(nn.Module):
    def __init__(self, wavelength, pixel_size, z, shape):
        super().__init__()
        H, W = shape
        fx = torch.fft.fftfreq(W, d=pixel_size)
        fy = torch.fft.fftfreq(H, d=pixel_size)
        FX, FY = torch.meshgrid(fx, fy, indexing='xy')
        term = (1.0 / wavelength)**2 - FX**2 - FY**2
        mask = (term > 0).float()
        phase = 2 * math.pi * z * torch.sqrt(term * mask)
        H_tf = torch.polar(mask, phase)
        self.register_buffer('H', H_tf)

    def forward(self, u):
        U = torch.fft.fft2(u, dim=(-2, -1))
        U_prop = U * self.H.unsqueeze(0).unsqueeze(0)
        return torch.fft.ifft2(U_prop, dim=(-2, -1))


class LenslessMicroscope(nn.Module):
    def __init__(
        self,
        led_positions,
        wavelengths,
        sensor_pixel_size=1.55e-6,
        upsample_factor=2,
        z_total=500e-6,         # Sample to sensor distance (500 micrometers)
        sensor_shape=(512, 512),
        sim_canvas_dim=1024,
    ):
        super().__init__()
        self.led_positions = led_positions
        self.wavelengths = wavelengths
        self.upsample_factor = upsample_factor
        self.sensor_pixel_size = sensor_pixel_size
        self.optical_pixel_size = sensor_pixel_size / upsample_factor
        self.sensor_shape = sensor_shape
        self.sim_opt_shape = (
            sim_canvas_dim * upsample_factor,
            sim_canvas_dim * upsample_factor,
        )
        self.z_total = z_total

        H_opt, W_opt = self.sim_opt_shape

        # Spatial Coordinates (Meters) on high-res optical grid
        x = torch.linspace(
            -W_opt / 2 * self.optical_pixel_size,
            W_opt / 2 * self.optical_pixel_size,
            W_opt,
            dtype=torch.float32,
        )
        y = torch.linspace(
            -H_opt / 2 * self.optical_pixel_size,
            H_opt / 2 * self.optical_pixel_size,
            H_opt,
            dtype=torch.float32,
        )
        X, Y = torch.meshgrid(x, y, indexing='xy')
        self.register_buffer('X_grid', X.unsqueeze(0).unsqueeze(0))
        self.register_buffer('Y_grid', Y.unsqueeze(0).unsqueeze(0))

        # Precomputed Transfer Functions for direct sample-to-sensor propagation
        self.prop_dict = nn.ModuleDict({
            ch: AngularSpectrumPropagator(
                wl, self.optical_pixel_size, self.z_total, self.sim_opt_shape
            )
            for ch, wl in wavelengths.items()
        })

    def forward(self, sensor_sample_field):
        H_s, W_s = self.sensor_shape
        H_c = self.sim_opt_shape[0] // self.upsample_factor
        W_c = self.sim_opt_shape[1] // self.upsample_factor

        pad_y = (H_c - H_s) // 2
        pad_x = (W_c - W_s) // 2

        # 1. Place sample surrounded by unity background (Real=1.0, Imag=0.0)
        real_pad = F.pad(
            sensor_sample_field.real,
            (pad_x, pad_x, pad_y, pad_y),
            mode='constant',
            value=1.0,
        )
        imag_pad = F.pad(
            sensor_sample_field.imag,
            (pad_x, pad_x, pad_y, pad_y),
            mode='constant',
            value=0.0,
        )

        real_up = F.interpolate(
            real_pad,
            scale_factor=self.upsample_factor,
            mode='bilinear',
            align_corners=False,
        )
        imag_up = F.interpolate(
            imag_pad,
            scale_factor=self.upsample_factor,
            mode='bilinear',
            align_corners=False,
        )
        stationary_sample = torch.complex(real_up, imag_up)

        sensor_list = []

        for ch, wl in self.wavelengths.items():
            prop = self.prop_dict[ch]

            for xL, yL, zL in self.led_positions:
                # 2. Spherical Wavefront Illumination
                R = torch.sqrt(
                    (self.X_grid - xL)**2 + (self.Y_grid - yL)**2 + zL**2
                )
                inc_field = (1.0 / R) * torch.exp(1j * (2 * math.pi / wl) * R)
                inc_field = inc_field / inc_field.abs().mean()

                # Step 1: Illumination strikes specimen
                field = stationary_sample * inc_field

                # Step 2: Direct propagation from Sample -> Sensor
                sensor_field = prop(field)
                high_res_intensity = (sensor_field.abs() ** 2).squeeze(1)

                # Step 3: Binning back to physical sensor pixel pitch via average pooling
                canvas_intensity = F.avg_pool2d(
                    high_res_intensity.unsqueeze(1),
                    kernel_size=self.upsample_factor,
                    stride=self.upsample_factor,
                ).squeeze(1)

                # Step 4: Geometric Projection Offset with Strict Bounds Clamping
                proj_x = -xL * (self.z_total / zL)
                proj_y = -yL * (self.z_total / zL)

                shift_x = int(round(proj_x / self.sensor_pixel_size))
                shift_y = int(round(proj_y / self.sensor_pixel_size))

                y_start = pad_y + shift_y
                x_start = pad_x + shift_x

                h_canvas, w_canvas = canvas_intensity.shape[-2], canvas_intensity.shape[-1]
                y_start = max(0, min(y_start, h_canvas - H_s))
                x_start = max(0, min(x_start, w_canvas - W_s))

                sensor_intensity = canvas_intensity[
                    ..., y_start : y_start + H_s, x_start : x_start + W_s
                ]

                # Safety fallback to guarantee exact (512, 512) shape
                if sensor_intensity.shape[-2] != H_s or sensor_intensity.shape[-1] != W_s:
                    sensor_intensity = F.interpolate(
                        sensor_intensity.unsqueeze(1), size=(H_s, W_s), mode='bilinear', align_corners=False
                    ).squeeze(1)

                sensor_list.append(sensor_intensity)

        return torch.stack(sensor_list, dim=1)


In [4]:
import os
import glob
import csv
import torch
import numpy as np
import torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
from tqdm import tqdm

# ============================================================================
# 1. Hardware Setup & Data Loader
# ============================================================================
def build_led_positions(pitch_x=8e-3, pitch_y=8e-3, n_led_x=2, n_led_y=2, distance_z=0.10):
    positions = []
    start_x = -(n_led_x - 1) * pitch_x / 2.0
    start_y = -(n_led_y - 1) * pitch_y / 2.0
    for i in range(n_led_x):
        for j in range(n_led_y):
            positions.append((start_x + i * pitch_x, start_y + j * pitch_y, distance_z))
    return positions

def load_single_npz_sample_with_raw(file_path, target_size=512):
    data = np.load(file_path)
    sensor_tensor = torch.from_numpy(data['input_tensor']).float()
    
    target_amp = torch.from_numpy(data.get('target_amp', data.get('amplitude'))).float().unsqueeze(0)
    target_phase = torch.from_numpy(data.get('target_phase', data.get('phase'))).float().unsqueeze(0)
    
    a_min, a_max = target_amp.min(), target_amp.max()
    if a_max - a_min > 1e-8:
        target_amp = (target_amp - a_min) / (a_max - a_min)
    else:
        target_amp = torch.zeros_like(target_amp)
        
    target_phase = torch.atan2(torch.sin(target_phase), torch.cos(target_phase))
    
    if sensor_tensor.shape[-1] != target_size or sensor_tensor.shape[-2] != target_size:
        sensor_tensor = F.interpolate(sensor_tensor.unsqueeze(0), size=(target_size, target_size), mode='bilinear', align_corners=False).squeeze(0)
        target_amp = F.interpolate(target_amp.unsqueeze(0), size=(target_size, target_size), mode='bilinear', align_corners=False).squeeze(0)
        target_phase = F.interpolate(target_phase.unsqueeze(0), size=(target_size, target_size), mode='bilinear', align_corners=False).squeeze(0)
        
    raw_sensor = sensor_tensor.clone()
    sensor_inputs_z = (sensor_tensor - sensor_tensor.mean(dim=(-2, -1), keepdim=True)) / (sensor_tensor.std(dim=(-2, -1), keepdim=True) + 1e-8)
                      
    return sensor_inputs_z.unsqueeze(0), raw_sensor.unsqueeze(0), target_amp.unsqueeze(0), target_phase.unsqueeze(0)

# ============================================================================
# 2. Three Distinct Classical Baselines (GS, HIO, GD-TV)
# ============================================================================
def iterative_gs_reconstruction(raw_sensor, forward_model, max_iterations=200, tol=1e-5):
    """Gerchberg-Saxton (GS) alternating projection with direct clipping."""
    device = raw_sensor.device
    b, c, h_s, w_s = raw_sensor.shape
    upsample = forward_model.upsample_factor
    h_c, w_c = forward_model.sim_opt_shape
    
    obj_complex = torch.ones((b, 1, h_c, w_c), dtype=torch.complex64, device=device)
    target_amp = torch.sqrt(torch.relu(raw_sensor) + 1e-8)
    
    prev_error = float('inf')
    for _ in range(max_iterations):
        channel_idx = 0
        current_error = 0.0
        
        for ch, wl in forward_model.wavelengths.items():
            prop = forward_model.prop_dict[ch]
            for xL, yL, zL in forward_model.led_positions:
                R = torch.sqrt((forward_model.X_grid - xL)**2 + (forward_model.Y_grid - yL)**2 + zL**2)
                inc_field = (1.0 / R) * torch.exp(1j * (2 * np.pi / wl) * R)
                inc_field = inc_field / inc_field.abs().mean()
                
                field_exit = obj_complex * inc_field
                sensor_field = prop(field_exit)
                
                proj_x = -xL * (forward_model.z_total / zL)
                proj_y = -yL * (forward_model.z_total / zL)
                shift_x = int(round(proj_x / forward_model.sensor_pixel_size))
                shift_y = int(round(proj_y / forward_model.sensor_pixel_size))
                
                pad_y = (h_c - h_s * upsample) // 2
                pad_x = (w_c - w_s * upsample) // 2
                y_start = pad_y + shift_y * upsample
                x_start = pad_x + shift_x * upsample
                
                sim_amp = sensor_field.abs()
                sim_phase = torch.angle(sensor_field)
                measured_amp_up = F.interpolate(target_amp[:, channel_idx:channel_idx+1], scale_factor=upsample, mode='bilinear', align_corners=False)
                
                current_error += F.l1_loss(sim_amp[..., y_start:y_start+(h_s*upsample), x_start:x_start+(w_s*upsample)], measured_amp_up).item()
                
                constrained_sensor_field = sensor_field.clone()
                constrained_sensor_field[..., y_start:y_start+(h_s*upsample), x_start:x_start+(w_s*upsample)] = \
                    measured_amp_up * torch.exp(1j * sim_phase[..., y_start:y_start+(h_s*upsample), x_start:x_start+(w_s*upsample)])
                
                inv_field_exit = torch.conj(prop(torch.conj(constrained_sensor_field)))
                obj_update = inv_field_exit / (inc_field + 1e-8)
                
                obj_amp = torch.clamp(torch.abs(obj_update), min=0.0, max=1.0)
                obj_phs = torch.angle(obj_update)
                obj_complex = obj_amp * torch.exp(1j * obj_phs)
                channel_idx += 1
                
        if abs(prev_error - current_error) < tol:
            break
        prev_error = current_error
                
    final_amp = F.avg_pool2d(torch.abs(obj_complex), kernel_size=upsample)
    final_phase = F.avg_pool2d(torch.angle(obj_complex), kernel_size=upsample)
    return final_amp, final_phase


def iterative_hio_reconstruction(raw_sensor, forward_model, max_iterations=200, beta=0.9, tol=1e-5):
    """Hybrid Input-Output (HIO) with beta feedback parameter."""
    device = raw_sensor.device
    b, c, h_s, w_s = raw_sensor.shape
    upsample = forward_model.upsample_factor
    h_c, w_c = forward_model.sim_opt_shape
    
    obj_complex = torch.ones((b, 1, h_c, w_c), dtype=torch.complex64, device=device)
    target_amp = torch.sqrt(torch.relu(raw_sensor) + 1e-8)
    
    prev_error = float('inf')
    for _ in range(max_iterations):
        obj_complex_prev = obj_complex.clone()
        channel_idx = 0
        current_error = 0.0
        
        for ch, wl in forward_model.wavelengths.items():
            prop = forward_model.prop_dict[ch]
            for xL, yL, zL in forward_model.led_positions:
                R = torch.sqrt((forward_model.X_grid - xL)**2 + (forward_model.Y_grid - yL)**2 + zL**2)
                inc_field = (1.0 / R) * torch.exp(1j * (2 * np.pi / wl) * R)
                inc_field = inc_field / inc_field.abs().mean()
                
                field_exit = obj_complex * inc_field
                sensor_field = prop(field_exit)
                
                proj_x = -xL * (forward_model.z_total / zL)
                proj_y = -yL * (forward_model.z_total / zL)
                shift_x = int(round(proj_x / forward_model.sensor_pixel_size))
                shift_y = int(round(proj_y / forward_model.sensor_pixel_size))
                
                pad_y = (h_c - h_s * upsample) // 2
                pad_x = (w_c - w_s * upsample) // 2
                y_start = pad_y + shift_y * upsample
                x_start = pad_x + shift_x * upsample
                
                sim_amp = sensor_field.abs()
                sim_phase = torch.angle(sensor_field)
                measured_amp_up = F.interpolate(target_amp[:, channel_idx:channel_idx+1], scale_factor=upsample, mode='bilinear', align_corners=False)
                
                current_error += F.l1_loss(sim_amp[..., y_start:y_start+(h_s*upsample), x_start:x_start+(w_s*upsample)], measured_amp_up).item()
                
                constrained_sensor_field = sensor_field.clone()
                constrained_sensor_field[..., y_start:y_start+(h_s*upsample), x_start:x_start+(w_s*upsample)] = \
                    measured_amp_up * torch.exp(1j * sim_phase[..., y_start:y_start+(h_s*upsample), x_start:x_start+(w_s*upsample)])
                
                inv_field_exit = torch.conj(prop(torch.conj(constrained_sensor_field)))
                obj_update = inv_field_exit / (inc_field + 1e-8)
                
                obj_amp = torch.abs(obj_update)
                obj_phs = torch.angle(obj_update)
                
                mask = (obj_amp > 1.0).float()
                hio_amp = obj_amp * (1 - mask) + (torch.abs(obj_complex_prev) - beta * obj_amp) * mask
                hio_amp = torch.clamp(hio_amp, min=0.0, max=1.0)
                
                obj_complex = hio_amp * torch.exp(1j * obj_phs)
                channel_idx += 1
                
        if abs(prev_error - current_error) < tol:
            break
        prev_error = current_error
                
    final_amp = F.avg_pool2d(torch.abs(obj_complex), kernel_size=upsample)
    final_phase = F.avg_pool2d(torch.angle(obj_complex), kernel_size=upsample)
    return final_amp, final_phase


def iterative_gdtv_reconstruction(raw_sensor, forward_model, max_iterations=200, lr=0.05, lambda_tv=0.005, tol=1e-5):
    """Gradient Descent with Total Variation (GD-TV) Regularization."""
    device = raw_sensor.device
    b, c, h_s, w_s = raw_sensor.shape
    
    # Initialize at base sensor canvas size (512x512)
    opt_amp = torch.zeros((b, 1, h_s, w_s), device=device, requires_grad=True) 
    opt_phase = torch.zeros((b, 1, h_s, w_s), device=device, requires_grad=True)
    
    optimizer = torch.optim.Adam([opt_amp, opt_phase], lr=lr)
    
    def tv_loss(img):
        dx = torch.abs(img[..., :-1] - img[..., 1:])
        dy = torch.abs(img[..., :-1, :] - img[..., 1:, :])
        return torch.mean(dx) + torch.mean(dy)
        
    prev_error = float('inf')
    for _ in range(max_iterations):
        optimizer.zero_grad()
        
        amp = torch.sigmoid(opt_amp)
        phase = torch.tanh(opt_phase) * np.pi
        obj_complex = amp.to(torch.complex64) * torch.exp(1j * phase.to(torch.complex64))
        
        sim_sensor = forward_model(obj_complex)
        if sim_sensor.shape[-2:] != raw_sensor.shape[-2:]:
            sim_sensor = F.interpolate(sim_sensor, size=raw_sensor.shape[-2:], mode='bilinear', align_corners=False)
            
        data_loss = F.mse_loss(sim_sensor, raw_sensor.float())
        reg_loss = tv_loss(amp) + 0.1 * tv_loss(phase)
        
        loss = data_loss + lambda_tv * reg_loss
        loss.backward()
        optimizer.step()
        
        if abs(prev_error - loss.item()) < tol:
            break
        prev_error = loss.item()
        
    final_amp = torch.sigmoid(opt_amp).detach()
    final_phase = (torch.tanh(opt_phase) * np.pi).detach()
    return final_amp, final_phase

# ============================================================================
# 3. Master Full-Dataset Evaluation Runner
# ============================================================================
def run_dataset_benchmark(dataset_dirs, forward_model, algorithm='hio', device='cuda', max_samples=None):
    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    forward_model.eval()
    
    algo_key = algorithm.lower().strip()
    if algo_key not in ['gs', 'hio', 'gd-tv']:
        raise ValueError(f"Unknown algorithm: '{algorithm}'. Choose from: 'gs', 'hio', 'gd-tv'")

    csv_file = f"Full_Dataset_Results_{algo_key.upper()}.csv"
    dataset_summary = {}

    print("\n" + "=" * 65)
    print(f" EVALUATING FULL TEST SET: {algo_key.upper()}")
    print(f" Output Destination: {csv_file}")
    print("=" * 65)

    for dataset_name, dir_path in dataset_dirs.items():
        files = sorted([f for f in glob.glob(os.path.join(dir_path, "*.npz")) if "scatterer_reference" not in os.path.basename(f)])
        
        if not files:
            print(f"[!] Warning: No .npz files found in {dir_path}. Skipping {dataset_name}.")
            continue
            
        if max_samples is not None:
            files = files[:max_samples]
            
        print(f"\n[*] Processing [{dataset_name}] ({len(files)} samples)...")
        
        a_psnr_list, a_ssim_list = [], []
        p_psnr_list, p_ssim_list = [], []
        
        for sample_path in tqdm(files, desc=f"{dataset_name}", leave=False):
            _, raw_sens, tgt_amp, tgt_phase = load_single_npz_sample_with_raw(sample_path)
            raw_sens = raw_sens.to(device)
            
            # Run selected baseline
            if algo_key == 'gs':
                with torch.no_grad():
                    pred_amp, pred_phase = iterative_gs_reconstruction(raw_sens, forward_model, max_iterations=50)
            elif algo_key == 'hio':
                with torch.no_grad():
                    pred_amp, pred_phase = iterative_hio_reconstruction(raw_sens, forward_model, max_iterations=50, beta=0.9)
            elif algo_key == 'gd-tv':
                with torch.enable_grad():
                    pred_amp, pred_phase = iterative_gdtv_reconstruction(raw_sens, forward_model, max_iterations=200, lr=0.05, lambda_tv=0.005)

            # Ensure strict shape matching
            if pred_amp.shape[-2:] != tgt_amp.shape[-2:]:
                pred_amp = F.interpolate(pred_amp, size=tgt_amp.shape[-2:], mode='bilinear', align_corners=False)
            if pred_phase.shape[-2:] != tgt_phase.shape[-2:]:
                pred_phase = F.interpolate(pred_phase, size=tgt_phase.shape[-2:], mode='bilinear', align_corners=False)

            gt_a = tgt_amp[0, 0].cpu().numpy()
            gt_p = tgt_phase[0, 0].cpu().numpy()
            pr_a = pred_amp[0, 0].detach().cpu().numpy()
            pr_p = pred_phase[0, 0].detach().cpu().numpy()
            
            # Metrics computation
            a_psnr = compute_psnr(gt_a, pr_a, data_range=1.0)
            a_ssim = compute_ssim(gt_a, pr_a, data_range=1.0)
            p_range = max(float(gt_p.max() - gt_p.min()), 1e-5)
            p_psnr = compute_psnr(gt_p, pr_p, data_range=p_range)
            p_ssim = compute_ssim(gt_p, pr_p, data_range=p_range)
            
            a_psnr_list.append(a_psnr)
            a_ssim_list.append(a_ssim)
            p_psnr_list.append(p_psnr)
            p_ssim_list.append(p_ssim)
            
        amp_psnr_stat = f"{np.mean(a_psnr_list):.2f} ± {np.std(a_psnr_list):.2f}"
        amp_ssim_stat = f"{np.mean(a_ssim_list):.4f} ± {np.std(a_ssim_list):.4f}"
        phs_psnr_stat = f"{np.mean(p_psnr_list):.2f} ± {np.std(p_psnr_list):.2f}"
        phs_ssim_stat = f"{np.mean(p_ssim_list):.4f} ± {np.std(p_ssim_list):.4f}"
        
        dataset_summary[dataset_name] = {
            'Amp PSNR': amp_psnr_stat,
            'Amp SSIM': amp_ssim_stat,
            'Phase PSNR': phs_psnr_stat,
            'Phase SSIM': phs_ssim_stat
        }
        
        print(f"[{dataset_name} Done]")
        print(f"  Amplitude -> PSNR: {amp_psnr_stat} dB | SSIM: {amp_ssim_stat}")
        print(f"  Phase     -> PSNR: {phs_psnr_stat} dB | SSIM: {phs_ssim_stat}")

    # Write summary CSV
    with open(csv_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Dataset", "Algorithm", "Amplitude PSNR (dB)", "Amplitude SSIM", "Phase PSNR (dB)", "Phase SSIM"])
        for d_name, stats in dataset_summary.items():
            writer.writerow([d_name, algo_key.upper(), stats['Amp PSNR'], stats['Amp SSIM'], stats['Phase PSNR'], stats['Phase SSIM']])
            
    print(f"\n[*] Summary written to: {os.path.abspath(csv_file)}")

# ============================================================================
# 4. Execution Block
# ============================================================================
if __name__ == "__main__":
    base_dir = "/home/sakshi/Unet 3+ reconstruction"
    
    dataset_dirs = {
        "Anemic RBCs": os.path.join(base_dir, "Digital_twin_simulation/Anemic_RGB"),
        "Malaria Smears": os.path.join(base_dir, "Digital_twin_simulation/Malaria_RGB"),
        "Rouleaux Aggregates": os.path.join(base_dir, "Digital_twin_simulation/Rauleaux_RGB"),
        "Lung Cancer Tissue": os.path.join(base_dir, "Digital_twin_simulation/Lung_cancer_RGB")
    }
    
    led_positions = build_led_positions(pitch_x=8e-3, pitch_y=8e-3, n_led_x=2, n_led_y=2, distance_z=0.10)
    wavelengths = {'R': 638e-9, 'G': 532e-9, 'B': 450e-9}
    
    forward_model = LenslessMicroscope(
        led_positions=led_positions, wavelengths=wavelengths,
        sensor_pixel_size=1.55e-6, upsample_factor=2, z_total=500e-6,
        sensor_shape=(512, 512), sim_canvas_dim=1024
    ).to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    
    # Select baseline: 'gs', 'hio', or 'gd-tv'
    CHOSEN_ALGORITHM = 'gs'
    
    run_dataset_benchmark(
        dataset_dirs=dataset_dirs,
        forward_model=forward_model,
        algorithm=CHOSEN_ALGORITHM,
        device='cuda',
        max_samples=None
    )


 EVALUATING FULL TEST SET: GS
 Output Destination: Full_Dataset_Results_GS.csv

[*] Processing [Anemic RBCs] (1000 samples)...


KeyboardInterrupt: 